In [1]:
# Colab setup: mount Drive and move into the project folder (edit PROJECT if your folder is elsewhere)
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/Colab Notebooks/careloop-no-show-guard"
    os.chdir(PROJECT)
except ImportError:
    pass                                   # not in Colab: stay in the current folder
print(os.getcwd()); print(os.listdir("."))

Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/careloop-no-show-guard
['02_queries.sql', 'data', 'notes_analysed.csv', '04_genai.ipynb', 'rejected_rows.json', 'clinics_enriched.csv', 'load_data.sql', 'appointments_clean.csv', '01_ingest_clean.ipynb', '02_run_queries_colab.ipynb', 'test_risk_scores.csv', '03_model.ipynb']


# 04 – GenAI: structured reasons from call-centre notes (Part 4)
**Never paste your key into a cell.** It is typed at run time into a hidden `getpass` box; clear the output and Edit → Find for the first characters of the key before downloading.

In [2]:
!pip install -q google-genai pydantic

**Model note:** `MODEL_NAME` must be a real Flash model name without the `models/` prefix. To list the ones your key can use, run: `for m in client.models.list(): print(m.name)` after creating `client`. If a call fails with an error about `thinking_config` / `thinking_level`, set `USE_THINKING = False` and re-run (the brief allows this).

In [12]:
import json, time, pandas as pd
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError

MODEL_NAME = "gemini-3-flash-preview"      # a Flash model from your own model list; see note below if it errors
USE_THINKING = True                        # set to False if the model rejects thinking_config
from getpass import getpass
API_KEY = getpass("Paste your key: ")      # typed at run time, hidden, never stored in the file
client = genai.Client(api_key=API_KEY)
CALLS = 0   # running count of API calls

ALLOWED = ["forgot_or_no_reminder","distance_or_transport","cost_or_fee","illness_or_emergency","work_or_schedule","other"]
SCHEMA = {"type": "object",
          "properties": {"root_cause": {"type": "string"}, "confidence": {"type": "number"},
                         "summary": {"type": "string"}, "recommended_action": {"type": "string"}},
          "required": ["root_cause", "confidence", "summary", "recommended_action"]}

SYSTEM_TEXT = ("You are a careful operations analyst for CareLoop Clinics. You read short call-centre notes "
               "and reply with JSON only, matching the requested schema.")

def craft_prompt(note_text):
    return f"""CONTEXT: CareLoop Clinics runs outpatient clinics in India. A call-centre agent phoned a patient who missed an appointment and wrote the note below. The note may be in English or in Hinglish (Hindi written in English letters).
ROLE: You are a healthcare operations analyst.
ACTION: Identify the single main reason the patient missed the appointment. root_cause must be exactly one of: {", ".join(ALLOWED)}. If the note is ambiguous or no listed cause clearly fits, answer "other" rather than guess. Also give confidence (0 to 1), a summary of at most 25 words, and one recommended_action for the clinic.
TONE: Neutral, concise, professional.
FORMAT: JSON with exactly the fields root_cause, confidence, summary, recommended_action.

NOTE: {note_text}"""

def call_model(prompt):
    """One API call; returns the raw JSON text exactly as returned."""
    global CALLS
    CALLS += 1
    cfg = dict(system_instruction=SYSTEM_TEXT, temperature=0, max_output_tokens=2000,
               response_mime_type="application/json", response_schema=SCHEMA)
    if USE_THINKING:
        cfg["thinking_config"] = types.ThinkingConfig(thinking_level="minimal")   # as in Session 22.1
    response = client.models.generate_content(model=MODEL_NAME, contents=prompt,
                                              config=types.GenerateContentConfig(**cfg))
    return response.text

LAST_RAW = ""
def analyse_note(note_text):
    """Send one note; returns the parsed dict. The raw reply text stays in LAST_RAW."""
    global LAST_RAW
    LAST_RAW = call_model(craft_prompt(note_text))
    return json.loads(LAST_RAW)

Paste your key: ··········


## Task 4.1 – one validated call (N001)

In [13]:
notes = pd.read_csv("data/followup_notes.csv")
first = notes.iloc[0]
print(first["note_id"], "|", first["note_text"])
print(analyse_note(first["note_text"]))

N001 | Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week.
{'root_cause': 'forgot_or_no_reminder', 'confidence': 1.0, 'summary': 'The patient missed the appointment because they did not receive a reminder and forgot the scheduled time.', 'recommended_action': 'Audit the automated SMS/email reminder system to ensure messages are being delivered successfully.'}


## Task 4.2 – validate and process the first 10 notes

In [14]:
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

def check_allowed(value):
    if value not in ALLOWED:
        raise ValueError(f"root_cause '{value}' not in allowed list")

results = []
for _, row in notes.head(10).iterrows():
    note_text = row["note_text"]
    rec = {"note_id": row["note_id"], "appointment_id": row["appointment_id"], "root_cause": "", "confidence": "",
           "summary": "", "recommended_action": "", "validation_status": "", "raw_response": ""}
    if pd.isna(note_text) or str(note_text).strip() == "":
        rec.update(root_cause="other", confidence=0.0, summary="No note recorded",
                   recommended_action="Re-call patient", validation_status="skipped_blank")
        results.append(rec); continue
    LAST_RAW = ""
    try:
        try:
            data = analyse_note(note_text)               # Task 4.2 says: call analyse_note
        finally:
            rec["raw_response"] = LAST_RAW               # keep the reply exactly as returned
        for k in ["root_cause", "confidence", "summary", "recommended_action"]:   # copy whatever fields exist, never repair
            if isinstance(data, dict) and k in data:
                rec[k] = data[k]
        insight = FollowupInsight(**data)
        check_allowed(insight.root_cause)
        rec["validation_status"] = "valid"
    except Exception as e:
        rec["validation_status"] = "invalid"
        print(row["note_id"], "invalid:", type(e).__name__, str(e)[:120])
    results.append(rec)
    time.sleep(13)                                  # free-tier rate limit

analysed = pd.DataFrame(results, columns=["note_id","appointment_id","root_cause","confidence","summary",
                                          "recommended_action","validation_status","raw_response"])
analysed.to_csv("notes_analysed.csv", index=False)
print(analysed[["note_id","root_cause","confidence","validation_status"]])
print("API calls so far:", CALLS)

  note_id             root_cause  confidence validation_status
0    N001  forgot_or_no_reminder        1.00             valid
1    N002  distance_or_transport        1.00             valid
2    N003            cost_or_fee        1.00             valid
3    N004                  other        0.00     skipped_blank
4    N005   illness_or_emergency        1.00             valid
5    N006       work_or_schedule        1.00             valid
6    N007  forgot_or_no_reminder        0.95             valid
7    N008  distance_or_transport        1.00             valid
8    N009                  other        0.90             valid
9    N010       work_or_schedule        1.00             valid
API calls so far: 10


In [15]:
expected = dict(zip([f"N{i:03d}" for i in range(1, 11)],
    ["forgot_or_no_reminder","distance_or_transport","cost_or_fee","other","illness_or_emergency",
     "work_or_schedule","forgot_or_no_reminder","distance_or_transport","other","work_or_schedule"]))
nonblank = analysed[analysed["validation_status"] != "skipped_blank"]
agree = sum(expected[r.note_id] == r.root_cause for r in nonblank.itertuples())
print(f"agreement with the answer key: {agree} of {len(nonblank)} non-blank notes (need >= 7 of 9)")

agreement with the answer key: 9 of 9 non-blank notes (need >= 7 of 9)


## Task 4.3 – break it on purpose

In [17]:
bad_replies = {"confidence = 1.7": {"root_cause": "other", "confidence": 1.7, "summary": "Patient was unwell", "recommended_action": "Call back"},
               "missing recommended_action": {"root_cause": "other", "confidence": 0.5, "summary": "Patient was unwell"}}
for label, reply in bad_replies.items():
    try:
        FollowupInsight(**reply)
    except ValidationError as e:
        print(f"--- {label} ---\n{e}\n")

--- confidence = 1.7 ---
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal

--- missing recommended_action ---
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'other', '...': 'Patient was unwell'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing



In [18]:
rows = []
for nid in ["N007", "N009"]:
    text = notes.loc[notes["note_id"] == nid, "note_text"].iloc[0]
    for version, prompt in [("V1", f"Classify this note: {text}"), ("V2", craft_prompt(text))]:
        raw = call_model(prompt)
        try:
            rc = json.loads(raw).get("root_cause")
        except Exception:
            rc = None
        rows.append({"note": nid, "prompt": version, "root_cause": rc, "in_allowed_list": rc in ALLOWED})
        time.sleep(13)
print(pd.DataFrame(rows))
print("TOTAL API calls used in Tasks 4.1-4.3:", CALLS)

   note prompt                root_cause  in_allowed_list
0  N007     V1  Missing SMS notification            False
1  N007     V2     forgot_or_no_reminder             True
2  N009     V1    Patient Unavailability            False
3  N009     V2                     other             True
TOTAL API calls used in Tasks 4.1-4.3: 14


**Deploy:** V2 (CRAFT + allowed values + 'answer other rather than guess'), because it returns an allowed label for the Hinglish and ambiguous notes. (Confirm against your own table.)

## Task 4.4 – merge and cross-tabulate

In [19]:
appts = pd.read_csv("appointments_clean.csv")
merged = analysed.merge(appts, on="appointment_id", how="left")
print(len(merged), "rows")
print(pd.crosstab(merged["root_cause"], merged["reminder_sent"]))
print("notes with reminder_sent = No:", (merged["reminder_sent"] == "No").sum(), "of", len(merged))

10 rows
reminder_sent          No  Yes
root_cause                    
cost_or_fee             0    1
distance_or_transport   1    1
forgot_or_no_reminder   2    0
illness_or_emergency    0    1
other                   1    1
work_or_schedule        1    1
notes with reminder_sent = No: 5 of 10
